# Lab 03 - Exploratory Data Analysis (EDA): Pima Indians Diabetes

Notebook truyền thống (Jupyter/Colab) được chuyển từ phiên bản marimo.

Scope:
- Data loading + schema validation
- Data quality checks (NaN / duplicates / zero-as-missing)
- Univariate / bivariate visualization
- Correlation analysis
- Insight tổng kết + hướng sang preprocessing/modeling
- MCP-assisted parity notes


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from IPython.display import Markdown, display

try:
    import ipywidgets as widgets
    WIDGETS_AVAILABLE = True
except Exception:
    widgets = None
    WIDGETS_AVAILABLE = False

plt.style.use('default')


In [ ]:
column_names = [
    'pregnancies',
    'glucose',
    'blood_pressure',
    'skin_thickness',
    'insulin',
    'bmi',
    'diabetes_pedigree_function',
    'age',
    'outcome',
]
feature_columns = [c for c in column_names if c != 'outcome']
zero_as_missing_columns = ['glucose', 'blood_pressure', 'skin_thickness', 'insulin', 'bmi']
random_state = 42


In [ ]:
def resolve_repo_root(start: Path) -> Path:
    candidates = [start, *start.parents]
    for p in candidates:
        if (p / 'Lab_03' / 'data' / 'pima-indians-diabetes.csv').exists():
            return p
    raise FileNotFoundError('Cannot find Lab_03/data/pima-indians-diabetes.csv from current working tree')

repo_root = resolve_repo_root(Path.cwd())
data_path = repo_root / 'Lab_03' / 'data' / 'pima-indians-diabetes.csv'
names_path = repo_root / 'Lab_03' / 'data' / 'pima-indians-diabetes.names'

print('repo_root =', repo_root)
print('data_path =', data_path)


In [ ]:
df = pd.read_csv(data_path, header=None, names=column_names)

display(Markdown('## 1) Data Loading & Problem Definition'))
display(Markdown(f'- Shape: **{df.shape[0]} rows x {df.shape[1]} columns**\n- Bài toán: **binary classification**\n- Target: `outcome` (0 = negative, 1 = positive diabetes)'))


In [ ]:
df.head(10)


In [ ]:
df.dtypes.to_frame(name='dtype')


In [ ]:
class_counts = df['outcome'].value_counts().sort_index().rename('count')
class_rate = df['outcome'].value_counts(normalize=True).sort_index().rename('rate')
class_distribution = (
    class_counts.to_frame()
    .join(class_rate.to_frame())
    .assign(rate_pct=lambda x: (100 * x['rate']).round(2))
)

display(Markdown('## 2) Class Distribution'))
class_distribution


In [ ]:
fig_class_dist, ax_class_dist = plt.subplots(figsize=(6, 4))
sns.barplot(
    x=class_distribution.index.astype(str),
    y=class_distribution['count'].values,
    ax=ax_class_dist,
    palette='viridis',
    hue=class_distribution.index.astype(str),
    legend=False,
)
ax_class_dist.set_title('Class distribution of outcome')
ax_class_dist.set_xlabel('Outcome')
ax_class_dist.set_ylabel('Count')
plt.tight_layout()
fig_class_dist


In [ ]:
missing_report = pd.DataFrame(
    {
        'nan_count': df.isna().sum(),
        'nan_rate_pct': (100 * df.isna().mean()).round(2),
        'zero_count': (df == 0).sum(),
        'zero_rate_pct': (100 * (df == 0).mean()).round(2),
    }
)
duplicate_count = int(df.duplicated().sum())
duplicates_preview = df[df.duplicated()].head(10)
zero_focus_report = (
    missing_report.loc[zero_as_missing_columns + ['pregnancies', 'outcome']]
    .sort_values('zero_rate_pct', ascending=False)
    .copy()
)

df_masked = df.copy()
df_masked[zero_as_missing_columns] = df_masked[zero_as_missing_columns].replace(0, np.nan)

print('Duplicate rows:', duplicate_count)


In [ ]:
display(Markdown('## 3) Data Quality Checks'))
missing_report


In [ ]:
display(Markdown('### Duplicate rows preview (nếu có)'))
duplicates_preview


In [ ]:
zero_focus_report


In [ ]:
if WIDGETS_AVAILABLE:
    bins_slider = widgets.IntSlider(min=10, max=80, step=1, value=30, description='bins')
    feature_selector = widgets.Dropdown(options=feature_columns, value='glucose', description='feature')
    mask_zero_toggle = widgets.Checkbox(value=True, description='mask zero')
    pairplot_toggle = widgets.Checkbox(value=False, description='pairplot')
    display(widgets.HBox([bins_slider, feature_selector, mask_zero_toggle, pairplot_toggle]))
else:
    bins_slider = None
    feature_selector = None
    mask_zero_toggle = None
    pairplot_toggle = None
    display(Markdown('ipywidgets chưa khả dụng: dùng fallback parameters ở cell tiếp theo.'))


In [ ]:
bins = 30
selected_feature = 'glucose'
use_zero_mask = True
enable_pairplot = False

if WIDGETS_AVAILABLE:
    bins = bins_slider.value
    selected_feature = feature_selector.value
    use_zero_mask = mask_zero_toggle.value
    enable_pairplot = pairplot_toggle.value

plot_df = df_masked if use_zero_mask else df

fig_hist, ax_hist = plt.subplots(figsize=(7, 4))
sns.histplot(data=plot_df, x=selected_feature, bins=bins, kde=True, ax=ax_hist, color='#4C72B0')
ax_hist.set_title(f'Distribution of {selected_feature}')
ax_hist.set_xlabel(selected_feature)
plt.tight_layout()
plt.show()

fig_box, ax_box = plt.subplots(figsize=(6, 4))
sns.boxplot(data=plot_df, x='outcome', y=selected_feature, ax=ax_box, palette='Set2', hue='outcome', legend=False)
ax_box.set_title(f'{selected_feature} by outcome')
ax_box.set_xlabel('outcome')
ax_box.set_ylabel(selected_feature)
plt.tight_layout()
plt.show()


In [ ]:
display(Markdown('## 4) Descriptive Statistics'))

display(df[feature_columns].describe().T)

grouped_stats = df.groupby('outcome')[feature_columns].agg(['mean', 'median', 'std']).round(3)
grouped_stats


In [ ]:
corr_source = df_masked if use_zero_mask else df
corr_matrix = corr_source.corr(numeric_only=True)

fig_corr, ax_corr = plt.subplots(figsize=(9, 7))
sns.heatmap(
    corr_matrix,
    cmap='coolwarm',
    center=0,
    annot=False,
    linewidths=0.4,
    cbar_kws={'shrink': 0.8},
    ax=ax_corr,
)
ax_corr.set_title('Correlation heatmap')
plt.tight_layout()
plt.show()

corr_matrix['outcome'].sort_values(ascending=False).to_frame(name='corr_with_outcome')


In [ ]:
fig_scatter, axes_scatter = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=corr_source, x='glucose', y='bmi', hue='outcome', alpha=0.7, ax=axes_scatter[0], palette='Set1')
axes_scatter[0].set_title('Glucose vs BMI by outcome')

sns.scatterplot(data=corr_source, x='glucose', y='insulin', hue='outcome', alpha=0.7, ax=axes_scatter[1], palette='Set1', legend=False)
axes_scatter[1].set_title('Glucose vs Insulin by outcome')

plt.tight_layout()
fig_scatter


In [ ]:
if enable_pairplot:
    pairplot_df = corr_source[['glucose', 'bmi', 'age', 'insulin', 'outcome']].dropna()
    grid = sns.pairplot(pairplot_df, hue='outcome', diag_kind='kde', corner=True)
    grid.fig.suptitle('Pairplot for selected features', y=1.02)
    plt.show()
else:
    print('Pairplot đang tắt (enable_pairplot=False hoặc widget pairplot=False).')


In [ ]:
top_zero_cols = zero_focus_report.index[:3].tolist()
summary_md = (
    f"- Dataset có **{int(class_distribution['count'].sum())}** quan sát, target lệch lớp vừa phải\n"
    f"  (class 0 khoảng **{class_distribution.loc[0, 'rate_pct']:.2f}%**, class 1 khoảng **{class_distribution.loc[1, 'rate_pct']:.2f}%**).\n"
    f"- Số dòng trùng lặp phát hiện: **{duplicate_count}**.\n"
    f"- Các cột có tỷ lệ `0` cao nhất (gợi ý missing ngầm): **{', '.join(top_zero_cols)}**.\n"
    "- Khi chuyển `0 -> NaN` ở các cột sinh lý (`glucose`, `blood_pressure`, `skin_thickness`, `insulin`, `bmi`),\n"
    "  ma trận tương quan và phân bố thường phản ánh dữ liệu thực tế hơn.\n"
    "- Bước tiếp theo: tách train/test có stratify, thử imputation pipeline và baseline model."
)

display(Markdown('## 5) EDA Insights Summary'))
display(Markdown(summary_md))


## 6) MCP-assisted Validation Workflow

Dùng MCP để kiểm chứng parity trước/sau chuyển đổi notebook:

1. `project-0-BackupSGU26_ML-pandas-mcp.read_metadata_tool`
   - Snapshot metadata (delimiter/encoding/row-count/column stats).
2. `project-0-BackupSGU26_ML-pandas-mcp.interpret_column_data`
   - Kiểm tra phân bố lớp cho cột nhãn (lưu ý tool này có thể suy diễn header từ dòng đầu CSV).
3. `project-0-BackupSGU26_ML-pandas-mcp.run_pandas_code_tool`
   - Đối chiếu số liệu cốt lõi: shape, class counts, zero counts, correlation summary.
4. `project-0-BackupSGU26_ML-mcp-server-ds.load_csv` + `run_script`
   - Dùng cho đối chiếu text/script (tool `run_script` không tạo chart).

Baseline tham chiếu đã đo bằng MCP code-path với schema chuẩn (`header=None`):
- shape: `(768, 9)`
- outcome counts: `{0: 500, 1: 268}`
- zero counts: `glucose=5, blood_pressure=35, skin_thickness=227, insulin=374, bmi=11`
